In [1]:
import os
from pathlib import Path
ROOT = "../" # Base directory relative to this notebook's location. Adjust if the notebook is moved
CSV_DIR = os.path.join(ROOT, "csv", "historical-data")

import pandas as pd
from IPython.display import clear_output

### Updating stats

In [2]:
def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def safe_stat(df, column, func):
    if column not in df.columns:
        return None

    return func(safe_numeric(df[column]))


rows = []

processed = 0

for filename in os.listdir(CSV_DIR):

    file_path = os.path.join(CSV_DIR, filename)

    # Ignore subfolders
    if not os.path.isfile(file_path):
        continue

    try:
        # Force indicativo as text
        df = pd.read_csv(file_path, dtype={"indicativo": str})

        clear_output(wait=True)
        print(f"Processing {filename}... ({processed + 1}/{len(os.listdir(CSV_DIR))})")

        rows.append(
            {
                "indicativo": str(df["indicativo"].iloc[0]) if "indicativo" in df.columns else None,
                "nombre": df["nombre"].iloc[0] if "nombre" in df.columns else None,
                "altitud": df["altitud"].iloc[0] if "altitud" in df.columns else None,
                "provincia": df["provincia"].iloc[0] if "provincia" in df.columns else None,
                "num_records": len(df),
                "avg_tmin": safe_stat(df, "tmin", lambda s: s.mean()),
                "avg_tmax": safe_stat(df, "tmax", lambda s: s.mean()),
                "avg_prec": safe_stat(df, "prec", lambda s: s.mean()),
                "avg_velmedia": safe_stat(df, "velmedia", lambda s: s.mean()),
                "std_tmed": safe_stat(df, "tmed", lambda s: s.std()),
            }
        )

        processed += 1

    except Exception as e:
        print(f"Error processing {filename}: {e}")

stations_df = pd.DataFrame(rows)

# Ensure consistent type before sorting
stations_df["indicativo"] = stations_df["indicativo"].astype(str)

stations_df = (
    stations_df
    .sort_values("indicativo")
    .reset_index(drop=True)
)

print(f"Processed {len(stations_df)} stations.")
display(stations_df.head())

Processing C939T_hist.csv... (768/770)
Processed 768 stations.


,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
0,0009X,ALFORJA,406,TARRAGONA,1760,10.326706,21.866588,1.275278,2.997434,6.333601
1,0016A,REUS AEROPUERTO,71,TARRAGONA,1950,11.727940,22.801129,1.312099,3.569625,6.325506
2,0034X,VALLS,233,TARRAGONA,1950,10.850077,22.453155,1.128718,NaN,6.341960
3,0042Y,TARRAGONA,55,TARRAGONA,1934,13.167755,22.431363,1.341408,NaN,5.705496
4,0061X,PONTONS,632,BARCELONA,1911,8.404505,19.591200,1.534488,3.449710,6.158788


In [62]:
stations_df.to_csv(os.path.join(ROOT, "csv", "stats", "weather-station-stats.csv"), index=False)

### Loading stats

In [3]:
stations_df = pd.read_csv(os.path.join(ROOT, "csv", "stats", "weather-station-stats.csv"))

### Descriptive analysis and validations

In [ ]:
# Highest Average High Temperature
highest_tmax = stations_df.sort_values("avg_tmax", ascending=False).head(15)
display(
    highest_tmax.style
    .set_properties(subset=["avg_tmax"], **{"background-color": "#ff9999", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
386,5514Z,GRANADA BASE AÉREA,695,GRANADA,1485,11.022811,26.961899,0.723028,3.324672,7.398497
742,C628B,"LA ALDEA DE SAN NICOLÁS, TASARTE",318,LAS PALMAS,1950,17.192204,26.949561,0.331298,3.919077,3.911497
406,5790Y,"SEVILLA, TABLADA",9,SEVILLA,1620,13.545730,26.834839,1.306117,1.843272,6.332577
399,5702X,CARMONA,50,SEVILLA,1781,12.687255,26.799712,1.398119,2.732022,6.687588
375,5361X,MONTORO,155,CORDOBA,1916,10.789443,26.653519,1.456227,1.493058,7.436028
396,5641X,ÉCIJA,130,SEVILLA,1939,12.402801,26.494767,1.315734,2.182128,7.020379
716,C319W,"VALLEHERMOSO, DAMA",190,STA. CRUZ DE TENERIFE,1951,16.812199,26.478934,0.368016,2.545231,2.768967
349,4541X,EL GRANADO,60,HUELVA,1959,12.172344,26.257812,1.220459,nan,6.379955
407,5796,MORÓN DE LA FRONTERA,87,SEVILLA,1951,12.719795,26.241077,1.491203,2.035457,6.714411
476,7178I,MURCIA,62,MURCIA,1951,14.048642,26.176269,0.924541,2.570771,6.576313


In [43]:
# Lowest Average High Temperature
lowest_tmax = stations_df.sort_values("avg_tmax", ascending=False).tail(10)
display(lowest_tmax.style.set_properties(subset=["avg_tmax"], **{"background-color": "#829cd4", "color": "black"}))

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
593,9445L,"FORMIGAL, SARRIOS",1800,HUESCA,1803,4.086832,11.246936,4.802456,nan,6.711358
537,9001S,ALTO CAMPOO,1650,CANTABRIA,1856,3.685179,11.160055,3.183259,3.203261,6.045328
594,9451F,"PANTICOSA, PETROSOS",1850,HUESCA,1945,4.417929,11.109381,3.646262,3.125750,6.794204
87,1221D,PAJARES-VALGRANDE,1480,ASTURIAS,1893,3.924867,11.072476,4.410551,2.953196,5.867952
634,9814I,"TORLA-ORDESA, EL CEBOLLAR",1905,HUESCA,1950,3.772858,10.998358,3.387566,2.919276,6.671582
72,1167G,"MIRADOR DEL CABLE, PARQUE NACIONAL PICOS DE EUROPA",1910,CANTABRIA,373,2.395148,8.139892,2.676829,5.366860,6.023616
636,9839V,"CERLER, COGULLA",2374,HUESCA,1956,1.318661,7.749233,2.317997,3.623402,6.594720
73,1167J,"CORISCAO, PARQUE NACIONAL PICOS DE EUROPA",1722,CANTABRIA,330,2.232508,7.576780,2.593403,3.265484,5.415132
653,9988B,CAP DE VAQUÈIRA,2467,LLEIDA,1928,0.160863,6.258733,nan,4.183624,6.921823
618,9677,PORT AINÉ,2410,LLEIDA,1887,0.372395,6.095489,nan,5.185623,6.936063


In [44]:
# Highest Average Low Temperature
highest_tmin = stations_df.sort_values("avg_tmin", ascending=False).head(10)

display(
    highest_tmin.style
    .set_properties(subset=["avg_tmin"], **{"background-color": "#f07d12", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
718,C329Z,SAN SEBASTIÁN DE LA GOMERA,15,STA. CRUZ DE TENERIFE,1964,19.782594,25.119452,0.358257,3.377789,2.908590
766,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1956,19.710838,23.813139,0.420661,6.555003,2.306746
708,C229J,PÁJARA,15,LAS PALMAS,1939,19.645003,25.237253,0.199895,3.251475,2.926524
755,C659M,"LAS PALMAS DE GRAN CANARIA, PL. DE LA FERIA",15,LAS PALMAS,1950,19.583297,23.715343,0.399169,2.014308,2.339005
729,C449C,STA.CRUZ DE TENERIFE,36,SANTA CRUZ DE TENERIFE,1800,19.560468,25.722871,0.517420,3.016215,2.863172
743,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1959,19.519642,23.754936,0.127535,3.023430,2.346820
738,C619X,AGAETE,5,LAS PALMAS,1956,19.405541,24.163725,0.281795,4.824591,2.656844
705,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1953,19.305598,24.736775,0.514710,6.374165,2.366184
767,C939T,"FRONTERA, SABINOSA",20,STA. CRUZ DE TENERIFE,1960,19.080490,23.663483,0.467776,3.213950,2.417431
746,C639M,"MASPALOMAS, C. INSULAR TURISMO",45,LAS PALMAS,1954,18.881048,25.762661,0.215670,2.752968,2.974474


In [45]:
# Highest Average Low Temperature excluding Canary Islands
highest_tmin_no_canary = stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)] \
    .sort_values("avg_tmin", ascending=False).head(10)
display(highest_tmin_no_canary.style.set_properties(subset=["avg_tmin"], **{"background-color": "#f07d12", "color": "black"}))

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
455,6329X,CABO DE GATA,42,ALMERIA,1940,17.626389,22.427831,0.394249,4.967166,4.907813
436,6083X,MARBELLA,2,MALAGA,1941,17.138334,22.476775,1.134307,4.054485,4.193661
676,B569X,CAPDEPERA,57,ILLES BALEARS,1917,16.935726,22.893461,1.320796,4.306075,5.495875
424,6000A,MELILLA,52,MELILLA,1960,16.622057,23.256244,0.745644,3.113316,4.846434
419,5973,CÁDIZ,2,CADIZ,1958,16.620706,22.505723,1.385444,4.519355,4.714792
442,6175X,RINCÓN DE LA VICTORIA,7,MALAGA,1929,16.412651,25.698028,1.024466,nan,5.082854
359,5000C,CEUTA,87,CEUTA,1950,16.348357,22.363501,2.196462,2.869583,4.508765
437,6088X,TORREMOLINOS,85,MALAGA,1956,16.224700,23.875091,1.143183,1.608919,5.154818
663,B228,"PALMA, PUERTO",3,ILLES BALEARS,1951,16.027567,23.421253,1.256556,1.962942,5.821579
451,6291B,EL EJIDO,98,ALMERIA,1961,15.904974,25.545855,0.283121,nan,5.377591


In [46]:
# Lowest Average Low Temperature
lowest_tmin = stations_df.sort_values("avg_tmin", ascending=False).tail(10)
display(lowest_tmin.style.set_properties(subset=["avg_tmin"], **{"background-color": "#cacbf1", "color": "black"}))

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
617,9657X,ESTERRI D'ÀNEU,952,LLEIDA,1928,3.237532,17.436886,1.649341,1.346782,6.287631
212,2766E,"SANABRIA, ROBLEDA-CERVANTES",933,ZAMORA,1956,2.780974,18.100000,2.580419,0.973900,5.878979
203,2630X,PUERTO DE SAN ISIDRO,1510,LEON,1664,2.471739,12.333737,2.682786,3.876034,5.897696
72,1167G,"MIRADOR DEL CABLE, PARQUE NACIONAL PICOS DE EUROPA",1910,CANTABRIA,373,2.395148,8.139892,2.676829,5.366860,6.023616
73,1167J,"CORISCAO, PARQUE NACIONAL PICOS DE EUROPA",1722,CANTABRIA,330,2.232508,7.576780,2.593403,3.265484,5.415132
612,9590D,CAP DE REC,1940,LLEIDA,1912,2.111432,11.359150,nan,1.536597,6.359448
264,3319D,PUERTO DEL PICO,1285,AVILA,1957,1.669646,16.723267,4.546848,2.477466,6.078949
636,9839V,"CERLER, COGULLA",2374,HUESCA,1956,1.318661,7.749233,2.317997,3.623402,6.594720
618,9677,PORT AINÉ,2410,LLEIDA,1887,0.372395,6.095489,nan,5.185623,6.936063
653,9988B,CAP DE VAQUÈIRA,2467,LLEIDA,1928,0.160863,6.258733,nan,4.183624,6.921823


In [47]:
# Highest Average Precipitation
highest_prec = stations_df.sort_values("avg_prec", ascending=False).head(10)
display(
    highest_prec.style
    .set_properties(subset=["avg_prec"], **{"background-color": "#a1cfff", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
123,1476R,"ROIS, CASAS DO PORTO",210,A CORUÑA,1937,10.137051,19.227581,8.967377,nan,4.726292
113,1406X,MAZARICOS,340,A CORUÑA,1944,9.491486,17.618998,8.650723,nan,4.539642
126,1489A,A LAMA,395,PONTEVEDRA,1944,8.441434,18.750310,7.388579,2.261368,5.188006
143,1696O,BEARIZ,610,OURENSE,1940,5.228883,19.492100,6.726098,nan,5.361435
120,1468X,A ESTRADA,269,PONTEVEDRA,1944,9.212519,19.643998,6.554127,nan,5.103344
413,5911A,GRAZALEMA,913,CADIZ,1924,10.117900,20.344678,6.444491,1.549738,6.548668
111,1399,VIMIANZO,287,A CORUÑA,1948,9.405095,17.838909,6.249275,nan,4.252636
127,1495,VIGO AEROPUERTO,255,PONTEVEDRA,1951,10.356540,19.367267,6.098723,3.495003,4.966618
147,1719,A CAÑIZA,560,PONTEVEDRA,1955,8.788775,18.312711,5.718272,nan,5.244065
39,1021X,"ERRENTERIA, AÑARBE",165,GIPUZKOA,1940,10.125838,19.949149,5.656669,1.556605,5.404259


In [48]:
# Lowest Avg Precipitation
lowest_prec = (
    stations_df.dropna(subset=["avg_prec"])
    .sort_values("avg_prec", ascending=True)
    .head(10)
)

display(
    lowest_prec.style
    .set_properties(subset=["avg_prec"], **{"background-color": "#e2d40e", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
743,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1959,19.519642,23.754936,0.127535,3.023430,2.346820
759,C689E,MASPALOMAS,6,LAS PALMAS,1955,18.375424,24.325989,0.160733,3.378699,2.805218
709,C239N,"TUINEJE,PUERTO GRAN TARAJAL",1,LAS PALMAS,1960,18.240245,25.273289,0.176982,4.050051,3.322895
699,C019V,YAIZA PLAYA BLANCA,6,LAS PALMAS,1947,18.362204,24.161946,0.187197,4.139548,2.803016
744,C629X,"MOGÁN, PUERTO",10,LAS PALMAS,1940,18.470775,25.358866,0.187250,3.068711,2.640014
711,C249I,FUERTEVENTURA AEROPUERTO,25,LAS PALMAS,1960,18.394493,24.543284,0.195838,6.168615,2.825682
708,C229J,PÁJARA,15,LAS PALMAS,1939,19.645003,25.237253,0.199895,3.251475,2.926524
746,C639M,"MASPALOMAS, C. INSULAR TURISMO",45,LAS PALMAS,1954,18.881048,25.762661,0.215670,2.752968,2.974474
760,C839X,LA GRACIOSA,19,LAS PALMAS,1872,18.158897,24.280513,0.241173,5.864115,2.624452
763,C919K,TACORON-LAPILLAS-TORTUGA,98,STA. CRUZ DE TENERIFE,1753,18.868021,25.066705,0.241991,3.334914,2.629190


In [49]:
# Lowest average precipitation excluding Canary Islands
lowest_prec_no_canary = (
    stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)]
    .dropna(subset=["avg_prec"])
    .sort_values("avg_prec", ascending=True)
    .head(10)
)
display(
    lowest_prec_no_canary.style.set_properties(subset=["avg_prec"], **{"background-color": "#e2d40e", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
451,6291B,EL EJIDO,98,ALMERIA,1961,15.904974,25.545855,0.283121,nan,5.377591
452,6293X,ROQUETAS DE MAR,3,ALMERIA,1758,14.122215,22.878553,0.327789,4.304025,5.043464
444,6205X,TORROX,3,MALAGA,1718,15.424954,23.309427,0.361354,3.210678,4.478553
455,6329X,CABO DE GATA,42,ALMERIA,1940,17.626389,22.427831,0.394249,4.967166,4.907813
454,6325O,ALMERÍA AEROPUERTO,21,ALMERIA,1956,15.762660,24.240307,0.520479,4.586769,5.581867
457,6364X,ALBOX,508,ALMERIA,1950,12.996546,25.186804,0.587427,3.464815,6.582294
456,6340X,GARRUCHA,28,ALMERIA,1958,15.334462,22.822103,0.625783,3.979254,5.337166
361,5047E,BAZA,785,GRANADA,1861,7.903313,23.788376,0.652485,3.057519,7.440422
460,7007Y,MAZARRÓN,66,MURCIA,1955,15.300924,24.876347,0.687410,3.232376,5.691339
480,7211B,PUERTO LUMBRERAS,445,MURCIA,1956,13.567763,24.525604,0.700383,nan,6.524131


In [50]:
# Highest average wind speed
highest_wind = stations_df.sort_values("avg_velmedia", ascending=False).head(10)
display(
    highest_wind.style
    .set_properties(subset=["avg_velmedia"], **{"background-color": "#ff99f7", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
104,1351,ESTACA DE BARES,90,A CORUÑA,1946,12.567962,16.859034,2.217276,8.473584,3.395864
750,C649I,GRAN CANARIA AEROPUERTO,24,LAS PALMAS,1951,18.771754,25.044568,0.391115,8.145473,2.797182
713,C314Z,"VALLEHERMOSO, ALTO IGUALERO",1474,STA. CRUZ DE TENERIFE,1934,10.436115,18.011694,1.470384,6.957210,6.102387
724,C430E,IZAÑA,2369,SANTA CRUZ DE TENERIFE,1440,7.467014,15.419444,0.557344,6.809843,5.900317
110,1393,CABO VILÁN,50,A CORUÑA,1922,11.957752,16.971391,3.387513,6.787197,3.187627
112,1400,FISTERRA,230,A CORUÑA,1950,11.846570,17.011449,2.968671,6.779914,3.687926
766,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1956,19.710838,23.813139,0.420661,6.555003,2.306746
705,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1953,19.305598,24.736775,0.514710,6.374165,2.366184
723,C429I,TENERIFE SUR AEROPUERTO,64,SANTA CRUZ DE TENERIFE,1956,18.187538,26.027408,0.322690,6.256511,3.006532
711,C249I,FUERTEVENTURA AEROPUERTO,25,LAS PALMAS,1960,18.394493,24.543284,0.195838,6.168615,2.825682


In [51]:
# Highest average wind speed excluding Canary Islands
highest_wind_no_canary = stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)] \
    .sort_values("avg_velmedia", ascending=False).head(10)
display(
    highest_wind_no_canary.style
    .set_properties(subset=["avg_velmedia"], **{"background-color": "#ff99f7", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
104,1351,ESTACA DE BARES,90,A CORUÑA,1946,12.567962,16.859034,2.217276,8.473584,3.395864
110,1393,CABO VILÁN,50,A CORUÑA,1922,11.957752,16.971391,3.387513,6.787197,3.187627
112,1400,FISTERRA,230,A CORUÑA,1950,11.846570,17.011449,2.968671,6.779914,3.687926
425,6001,TARIFA,32,CADIZ,1948,15.135252,20.714851,1.886361,6.155621,4.254151
671,B398A,"CABRERA, PARQUE NACIONAL DE CABRERA",165,BALEARES,422,14.969359,19.690974,0.957820,5.688333,5.385352
190,2491C,"LA COVATILLA, ESTACIÓN DE ESQUÍ",1960,SALAMANCA,1874,3.986069,11.421658,4.032533,5.388626,6.767167
72,1167G,"MIRADOR DEL CABLE, PARQUE NACIONAL PICOS DE EUROPA",1910,CANTABRIA,373,2.395148,8.139892,2.676829,5.366860,6.023616
604,9550C,"ANDORRA, HORCALLANA",762,TERUEL,1937,9.440930,19.729580,0.916308,5.306909,7.196612
607,9563X,CASTELLFORT,1220,CASTELLON,1899,8.008908,16.111347,1.507040,5.250000,6.927773
618,9677,PORT AINÉ,2410,LLEIDA,1887,0.372395,6.095489,nan,5.185623,6.936063


In [52]:
# Lowest average wind speed
lowest_wind = (
    stations_df.dropna(subset=["avg_velmedia"])
    .sort_values("avg_velmedia", ascending=True)
    .head(10)
)
display(
    lowest_wind.style
    .set_properties(subset=["avg_velmedia"], **{"background-color": "#99ff99", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
427,6040X,CORTES DE LA FRONTERA,315,MALAGA,1919,11.926600,23.561438,2.590456,0.709183,6.015378
24,0360X,LES PLANES D'HOSTOLES,337,GIRONA,1931,6.727268,21.500209,2.104973,0.759206,6.567517
144,1700X,O CARBALLIÑO,400,OURENSE,1942,7.450000,20.503021,3.445534,0.785044,5.895863
661,B103B,ANDRATX - SANT ELM,52,BALEARES,1933,13.120084,22.835310,1.069375,0.855898,5.934885
595,9453X,"BIESCAS, EMBALSE DE BÚBAL",1100,HUESCA,1960,4.672699,16.557077,4.111225,0.870787,6.494747
388,5515X,GRANADA-CARTUJA,775,GRANADA,1940,10.954474,24.829723,0.902431,0.903026,7.556050
274,3423I,MADRIGAL DE LA VERA,464,CACERES,1960,11.186821,22.931436,3.644410,0.906605,7.409729
658,B013X,"ESCORCA, LLUC",490,ILLES BALEARS,1951,9.725748,21.192879,2.819458,0.938779,6.406004
637,9843A,SEIRA,825,HUESCA,1956,5.840153,19.437769,2.951509,0.953476,6.965728
212,2766E,"SANABRIA, ROBLEDA-CERVANTES",933,ZAMORA,1956,2.780974,18.100000,2.580419,0.973900,5.878979


In [57]:
# Highest Temperature Variability (std of tmed)
highest_variability = stations_df.sort_values("std_tmed", ascending=False).head(10)
display(
    highest_variability.style
    .set_properties(subset=["std_tmed"], **{"background-color": "#c80dda", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
300,4064Y,ALCAZAR DE SAN JUAN,640,CIUDAD REAL,1943,10.613967,23.016917,0.901867,1.623569,8.333520
616,9650X,ARTESA DE SEGRE,400,LLEIDA,1951,7.652464,21.474692,1.091812,nan,8.076644
314,4147X,VALDEPEÑAS,700,CIUDAD REAL,1944,10.027324,22.771864,0.895273,1.815396,8.068184
239,3085Y,PASTRANA,920,GUADALAJARA,1829,7.858475,20.707627,1.536581,nan,8.062121
315,4148,VISO DEL MARQUÉS,804,CIUDAD REAL,1871,9.381059,22.340128,1.216122,2.742460,8.039264
312,4121,CIUDAD REAL,626,CIUDAD REAL,1960,10.560245,22.927477,1.153240,2.104972,8.037380
360,5038Y,CAZORLA,799,JAEN,1952,12.634857,23.633455,1.548975,nan,8.034260
310,4103X,TOMELLOSO,662,CIUDAD REAL,1894,9.245445,23.196590,0.908445,2.774054,8.026180
311,4116I,ALMAGRO / FAMET,626,CIUDAD REAL,1965,8.731202,23.019417,1.024432,2.883693,8.007454
621,9707,LLIMIANA,515,LLEIDA,1910,7.441777,22.236277,1.533828,1.816632,8.002564


In [32]:
# Lowest Temperature Variability (std of tmed)
lowest_variability = (
    stations_df.dropna(subset=["std_tmed"])
    .sort_values("std_tmed", ascending=True)
    .head(10)
)
display(
    lowest_variability.style.set_properties(subset=["std_tmed"], **{"background-color": "#cbb0cc", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
766,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1956,19.710838,23.813139,0.420661,6.555003,2.306746
755,C659M,"LAS PALMAS DE GRAN CANARIA, PL. DE LA FERIA",15,LAS PALMAS,1950,19.583297,23.715343,0.399169,2.014308,2.339005
754,C659H,"LAS PALMAS DE GRAN CANARIA, SAN CRISTOBAL",55,LAS PALMAS,1934,18.397410,23.692364,0.356768,3.954835,2.344277
743,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1959,19.519642,23.754936,0.127535,3.023430,2.346820
705,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1953,19.305598,24.736775,0.514710,6.374165,2.366184
767,C939T,"FRONTERA, SABINOSA",20,STA. CRUZ DE TENERIFE,1960,19.080490,23.663483,0.467776,3.213950,2.417431
706,C139E,LA PALMA AEROPUERTO,33,SANTA CRUZ DE TENERIFE,1956,18.749846,23.493859,0.812104,5.336280,2.423936
758,C669B,ARUCAS,86,LAS PALMAS,1951,17.926756,23.583006,0.450880,2.531709,2.464945
733,C459Z,PUERTO DE LA CRUZ,25,STA. CRUZ DE TENERIFE,1955,18.490685,24.706794,0.756909,2.673468,2.482047
760,C839X,LA GRACIOSA,19,LAS PALMAS,1872,18.158897,24.280513,0.241173,5.864115,2.624452


In [33]:
# Lowest temperature variability excluding Canary Islands
lowest_variability_no_canary = (
    stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)]
    .dropna(subset=["std_tmed"])
    .sort_values("std_tmed", ascending=True)
    .head(10)
)
display(
    lowest_variability_no_canary.style.set_properties(subset=["std_tmed"], **{"background-color": "#cbb0cc", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
110,1393,CABO VILÁN,50,A CORUÑA,1922,11.957752,16.971391,3.387513,6.787197,3.187627
104,1351,ESTACA DE BARES,90,A CORUÑA,1946,12.567962,16.859034,2.217276,8.473584,3.395864
112,1400,FISTERRA,230,A CORUÑA,1950,11.846570,17.011449,2.968671,6.779914,3.687926
106,1387,A CORUÑA,57,A CORUÑA,1951,12.671861,18.752025,3.218740,3.603910,3.778618
103,1347T,BURELA,80,LUGO,1948,12.127609,17.753111,2.522222,nan,3.810056
107,1387D,A CORUÑA BENS,132,A CORUÑA,1922,11.715951,17.742532,3.163237,4.240031,3.827505
95,1283U,CABO BUSTO,60,ASTURIAS,1870,11.712257,17.837203,1.941193,4.746667,3.875812
85,1210X,CABO PEÑAS,100,ASTURIAS,1851,12.596631,17.550084,2.418028,3.978088,3.899482
101,1342X,RIBADEO,43,LUGO,1925,11.127016,18.531132,2.458885,3.154943,4.053713
84,1208H,"GIJÓN, PUERTO",5,ASTURIAS,1951,12.697793,18.866838,2.644912,nan,4.067084


In [64]:
# Highest daily temperature range (avg_tmax - avg_tmin)
stations_df["avg_temp_range"] = stations_df["avg_tmax"] - stations_df["avg_tmin"]
highest_temp_range = stations_df.sort_values("avg_temp_range", ascending=False).head(10)
display(
    highest_temp_range.style
    .set_properties(subset=["avg_temp_range"], **{"background-color": "#e9430c", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,avg_temp_range
168,2192C,CUÉLLAR,795,SEGOVIA,1954,3.288672,20.736322,1.377909,2.546465,6.831711,17.447649
389,5530E,GRANADA AEROPUERTO,560,GRANADA,1957,8.630895,25.095601,1.020247,2.640174,7.550015,16.464706
397,5654X,LA PUEBLA DE LOS INFANTES,200,SEVILLA,1952,8.965531,25.415333,2.003828,1.874650,6.987207,16.449802
611,9590,MARTINET,1038,LLEIDA,1960,3.451507,19.851354,1.703459,1.609699,7.050344,16.399847
510,8245Y,MIRA,815,CUENCA,1396,5.551400,21.815506,1.070173,nan,7.004783,16.264106
521,8381X,ADEMUZ,705,VALENCIA,1960,6.230930,22.320603,1.048571,nan,7.165699,16.089673
386,5514Z,GRANADA BASE AÉREA,695,GRANADA,1485,11.022811,26.961899,0.723028,3.324672,7.398497,15.939088
166,2172Y,SARDÓN DE DUERO,725,VALLADOLID,1903,5.029984,20.962209,1.199730,2.251812,7.012562,15.932225
361,5047E,BAZA,785,GRANADA,1861,7.903313,23.788376,0.652485,3.057519,7.440422,15.885062
375,5361X,MONTORO,155,CORDOBA,1916,10.789443,26.653519,1.456227,1.493058,7.436028,15.864076


In [59]:
# Lowest daily temperature range (avg_tmax - avg_tmin)
lowest_temp_range = (
    stations_df.dropna(subset=["avg_temp_range"])
    .sort_values("avg_temp_range", ascending=True)
    .head(10)
)
display(
    lowest_temp_range.style.set_properties(subset=["avg_temp_range"], **{"background-color": "#99e699", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,avg_temp_range
766,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1956,19.710838,23.813139,0.420661,6.555003,2.306746,4.102301
755,C659M,"LAS PALMAS DE GRAN CANARIA, PL. DE LA FERIA",15,LAS PALMAS,1950,19.583297,23.715343,0.399169,2.014308,2.339005,4.132046
743,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1959,19.519642,23.754936,0.127535,3.023430,2.346820,4.235294
104,1351,ESTACA DE BARES,90,A CORUÑA,1946,12.567962,16.859034,2.217276,8.473584,3.395864,4.291071
34,0433D,CABO DE CREUS,75,GIRONA,1891,15.594831,20.130669,1.003132,nan,5.581685,4.535838
767,C939T,"FRONTERA, SABINOSA",20,STA. CRUZ DE TENERIFE,1960,19.080490,23.663483,0.467776,3.213950,2.417431,4.582993
671,B398A,"CABRERA, PARQUE NACIONAL DE CABRERA",165,BALEARES,422,14.969359,19.690974,0.957820,5.688333,5.385352,4.721615
706,C139E,LA PALMA AEROPUERTO,33,SANTA CRUZ DE TENERIFE,1956,18.749846,23.493859,0.812104,5.336280,2.423936,4.744012
738,C619X,AGAETE,5,LAS PALMAS,1956,19.405541,24.163725,0.281795,4.824591,2.656844,4.758184
455,6329X,CABO DE GATA,42,ALMERIA,1940,17.626389,22.427831,0.394249,4.967166,4.907813,4.801442


In [60]:
# Lowest daily temperature range (avg_tmax - avg_tmin) excluding Canary Islands
lowest_temp_range_no_canary = (
    stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)]
    .dropna(subset=["avg_temp_range"])
    .sort_values("avg_temp_range", ascending=True)
    .head(10)
)
display(
    lowest_temp_range_no_canary.style.set_properties(subset=["avg_temp_range"], **{"background-color": "#99e699", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,avg_temp_range
104,1351,ESTACA DE BARES,90,A CORUÑA,1946,12.567962,16.859034,2.217276,8.473584,3.395864,4.291071
34,0433D,CABO DE CREUS,75,GIRONA,1891,15.594831,20.130669,1.003132,nan,5.581685,4.535838
671,B398A,"CABRERA, PARQUE NACIONAL DE CABRERA",165,BALEARES,422,14.969359,19.690974,0.957820,5.688333,5.385352,4.721615
455,6329X,CABO DE GATA,42,ALMERIA,1940,17.626389,22.427831,0.394249,4.967166,4.907813,4.801442
85,1210X,CABO PEÑAS,100,ASTURIAS,1851,12.596631,17.550084,2.418028,3.978088,3.899482,4.953453
110,1393,CABO VILÁN,50,A CORUÑA,1922,11.957752,16.971391,3.387513,6.787197,3.187627,5.013639
112,1400,FISTERRA,230,A CORUÑA,1950,11.846570,17.011449,2.968671,6.779914,3.687926,5.164879
53,1057B,MATXITXAKO,93,BIZKAIA,1951,12.989092,18.253758,3.442334,5.147309,4.505290,5.264666
65,1111X,SANTANDER,51,CANTABRIA,1951,12.786697,18.119671,3.447652,4.621498,4.228021,5.332974
436,6083X,MARBELLA,2,MALAGA,1941,17.138334,22.476775,1.134307,4.054485,4.193661,5.338441
